In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
# *** NOVAS IMPORTAÇÕES ***
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression # O nosso "Meta-Modelo"
from sklearn.model_selection import cross_val_score
import warnings
import time 

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')
warnings.filterwarnings('ignore', category=Warning) 

print("--- [Início do Pipeline (v16 - Stacking de Elite)] ---")
print("A combinar XGBoost (v11) e LightGBM (v12) com um Meta-Modelo...")

# --- 1. FUNÇÃO DE PRÉ-PROCESSAMENTO (IDÊNTICA AO v11) ---

def formatar_celula(series_coluna):
    s = series_coluna.astype(str).replace('NULL', pd.NA)
    s = s.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    s = s.str.upper()
    s = s.str.replace(r'[^A-Z0-9]+', '_', regex=True)
    s = s.str.strip('_')
    s = s.replace('', pd.NA)
    return s

def preprocessar_dados(df, colunas_scaler_treinadas=None, scaler=None):
    if 'RowId' not in df.columns and 'AVERAGE_SPEED_DIFF' not in df.columns:
        df_final_row_ids = np.arange(1, len(df) + 1)
    else:
        df_final_row_ids = None

    cols_to_transform = ['AVERAGE_CLOUDINESS', 'AVERAGE_RAIN']
    for col in cols_to_transform:
        if col in df.columns:
            df[col] = formatar_celula(df[col])

    cols_to_drop_base = ['city_name', 'AVERAGE_RAIN', 'AVERAGE_PRECIPITATION', 'record_date']
    
    try:
        df['record_date_dt'] = pd.to_datetime(df['record_date'], format='mixed', dayfirst=True)
        df['Hora_sin'] = np.sin(2 * np.pi * df['record_date_dt'].dt.hour/24)
        df['Hora_cos'] = np.cos(2 * np.pi * df['record_date_dt'].dt.hour/24)
        df['Mes_sin'] = np.sin(2 * np.pi * df['record_date_dt'].dt.month/12)
        df['Mes_cos'] = np.cos(2 * np.pi * df['record_date_dt'].dt.month/12)
        df['DIA_SEMANA'] = df['record_date_dt'].dt.dayofweek
        df['IS_WEEKEND'] = df['DIA_SEMANA'].isin([5, 6]).astype(int)
        rush_hours = [7, 8, 9, 17, 18, 19]
        df['IS_RUSH_HOUR'] = df['record_date_dt'].dt.hour.isin(rush_hours).astype(int)
        df = df.drop(columns=['record_date_dt'])
    except KeyError:
        pass

    cols_existentes_drop = [col for col in cols_to_drop_base if col in df.columns]
    df = df.drop(columns=cols_existentes_drop)

    if 'AVERAGE_CLOUDINESS' in df.columns:
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('NAN', np.nan)
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].fillna('NONE')

    cols_to_onehot = ['LUMINOSITY', 'AVERAGE_CLOUDINESS', 'DIA_SEMANA']
    for col in cols_to_onehot:
        if col in df.columns:
            prefix = col[:3].upper()
            if col == 'DIA_SEMANA': prefix = 'DAY'
            dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int, sparse=False)
            df = pd.concat([df, dummies], axis=1)
            df = df.drop(col, axis=1)

    cols_to_normalize = [
        'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME',
        'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY',
        'AVERAGE_WIND_SPEED', 'IS_WEEKEND', 'IS_RUSH_HOUR',
        'Hora_sin', 'Hora_cos', 'Mes_sin', 'Mes_cos'
    ]
    cols_existentes_normalize = [col for col in cols_to_normalize if col in df.columns]
    
    if scaler is None:
        print("... (TREINO) A fitar o MinMaxScaler.")
        scaler = MinMaxScaler()
        if cols_existentes_normalize:
            df[cols_existentes_normalize] = scaler.fit_transform(df[cols_existentes_normalize])
        return df, scaler, cols_existentes_normalize, df_final_row_ids
    else:
        print("... (TESTE) A aplicar o MinMaxScaler já treinado.")
        cols_para_scaler_teste = [col for col in colunas_scaler_treinadas if col in df.columns]
        if cols_para_scaler_teste:
            df[cols_para_scaler_teste] = scaler.transform(df[cols_para_scaler_teste])
        return df, None, None, df_final_row_ids


# --- 2. CARREGAR DADOS ---
print("\n--- [Passo 1/7] A carregar dados de Treino e Teste... ---")
df_train = pd.read_csv("training_data.csv", delimiter=";", encoding="latin-1")
df_test = pd.read_csv("test_data.csv", delimiter=",", encoding="latin-1")
y_train_raw = df_train.pop('AVERAGE_SPEED_DIFF')
test_row_ids = np.arange(1, len(df_test) + 1)


# --- 3. PRÉ-PROCESSAMENTO (TREINO) ---
print("--- [Passo 2/7] A pré-processar dados de TREINO... ---")
X_train, scaler, colunas_scaler, _ = preprocessar_dados(df_train)
print("... (TREINO) A fitar o LabelEncoder.")
le = LabelEncoder()
y_train_formatado = formatar_celula(y_train_raw).replace('NAN', 'NONE').fillna('NONE')
y_train_encoded = le.fit_transform(y_train_formatado)
print(f"Classes do Alvo encontradas (em MAIÚSCULAS): {list(le.classes_)}")


# --- 4. PRÉ-PROCESSAMENTO (TESTE) ---
print("--- [Passo 3/7] A pré-processar dados de TESTE... ---")
X_test, _, _, _ = preprocessar_dados(df_test, colunas_scaler_treinadas=colunas_scaler, scaler=scaler)


# --- 5. ALINHAMENTO DE COLUNAS ---
print("--- [Passo 4/7] A alinhar colunas de Treino e Teste... ---")
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_train = X_train.reindex(columns=X_test.columns, fill_value=0)
print(f"Shape final do Treino (X_train): {X_train.shape}")
print(f"Shape final do Teste (X_test):  {X_test.shape}")
print(f"Número de features: {X_train.shape[1]}") # 33


# --- 6. VALIDAÇÃO DO ENSEMBLE (Stacking) ---
print("\n--- [Passo 5/7] A configurar e validar o StackingClassifier... ---")

# Modelos de Base (Nível 0)
estimators = [
    ('xgb', XGBClassifier(
        learning_rate=0.08, max_depth=5, n_estimators=100,
        subsample=0.7, colsample_bytree=0.9,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=42, n_jobs=-1
    )),
    ('lgbm', LGBMClassifier(
        subsample=0.9, num_leaves=50, n_estimators=200,
        learning_rate=0.01, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    ))
]

# Meta-Modelo (Nível 1)
# Um modelo simples que aprende com as previsões dos modelos de base
meta_model = LogisticRegression(random_state=42, n_jobs=-1)

# Criar o Stack
# cv=5 significa que as previsões do Nível 0 são feitas "out-of-fold"
# para evitar que o meta-modelo "cole" dos dados de treino.
stack_clf = StackingClassifier(
    estimators=estimators, 
    final_estimator=meta_model, 
    cv=5, # Crucial
    n_jobs=-1
)

print("A calcular o Cross-Validation do Stack (isto vai demorar MUITO)...")
start_time = time.time()
# O Stacking já tem CV interno, mas vamos correr um CV externo para sermos justos
scores = cross_val_score(stack_clf, X_train, y_train_encoded, cv=5, scoring='accuracy', n_jobs=1, verbose=1)
end_time = time.time()

print(f"\nValidação concluída em {(end_time - start_time) / 60:.2f} minutos.")
print(f"Score ANTERIOR (XGBoost v11): 0.81283")
print(f"MELHOR ACCURACY (Stacking v16): {np.mean(scores):.5f}")


# --- 7. TREINO DO MODELO FINAL (STACKING) ---
print("\n--- [Passo 6/7] A treinar o modelo FINAL (Stacking)... ---")
stack_clf.fit(X_train, y_train_encoded)
print("... Modelo final treinado com sucesso!")


# --- 8. PREVISÃO E SUBMISSÃO ---
print("--- [Passo 7/7] A gerar previsões e a criar submission.csv... ---")
y_pred_encoded = stack_clf.predict(X_test)
y_pred_labels_upper = le.inverse_transform(y_pred_encoded)
y_pred_labels_final = pd.Series(y_pred_labels_upper).str.title()

submission_df = pd.DataFrame({
    'RowId': test_row_ids,
    'Speed_Diff': y_pred_labels_final
})

submission_df.to_csv('submission_v16_stacking.csv', index=False)

print("\n--- [FIM DO PIPELINE] ---")
print("Ficheiro 'submission_v16_stacking.csv' criado com sucesso!")
print(submission_df.head())

--- [Início do Pipeline (v16 - Stacking de Elite)] ---
A combinar XGBoost (v11) e LightGBM (v12) com um Meta-Modelo...

--- [Passo 1/7] A carregar dados de Treino e Teste... ---
--- [Passo 2/7] A pré-processar dados de TREINO... ---
... (TREINO) A fitar o MinMaxScaler.
... (TREINO) A fitar o LabelEncoder.
Classes do Alvo encontradas (em MAIÚSCULAS): ['HIGH', 'LOW', 'MEDIUM', 'NONE', 'VERY_HIGH']
--- [Passo 3/7] A pré-processar dados de TESTE... ---
... (TESTE) A aplicar o MinMaxScaler já treinado.
--- [Passo 4/7] A alinhar colunas de Treino e Teste... ---
Shape final do Treino (X_train): (6812, 33)
Shape final do Teste (X_test):  (1500, 33)
Número de features: 33

--- [Passo 5/7] A configurar e validar o StackingClassifier... ---
A calcular o Cross-Validation do Stack (isto vai demorar MUITO)...


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  2.9min finished



Validação concluída em 2.89 minutos.
Score ANTERIOR (XGBoost v11): 0.81283
MELHOR ACCURACY (Stacking v16): 0.81489

--- [Passo 6/7] A treinar o modelo FINAL (Stacking)... ---
... Modelo final treinado com sucesso!
--- [Passo 7/7] A gerar previsões e a criar submission.csv... ---

--- [FIM DO PIPELINE] ---
Ficheiro 'submission_v16_stacking.csv' criado com sucesso!
   RowId Speed_Diff
0      1       None
1      2     Medium
2      3       None
3      4       High
4      5        Low
